# 23 · Graph and Geometric Deep Learning

- **Level:** Modern (2016 → present)
- **Typical semester:** research and electives
- **Prerequisites:** 06, 09 (§7 graph Fourier), 13 (§8 graph matrices), 18
- **Matches code in:** `lipipy.dl` (graph layers) and `lipipy.research` (spatiotemporal, equivariant)


## 1. Spectral graph convolution

Graph Fourier transform with $L = U\Lambda U^\top$: $\hat x = U^\top x$. Spectral filter: $g_\theta * x = Ug_\theta(\Lambda)U^\top x$.

**ChebNet:** $g_\theta(\Lambda)\approx\sum_{k=0}^K\theta_kT_k(\tilde\Lambda)$, $\tilde\Lambda = \frac{2}{\lambda_{\max}}\Lambda - I$; Chebyshev recurrence $T_k = 2xT_{k-1} - T_{k-2}$ ($K$-hop localized, no eigendecomposition)


## 2. Graph convolutional network (GCN)

$$H^{(l+1)} = \sigma\left(\tilde D^{-1/2}\tilde A\tilde D^{-1/2}H^{(l)}W^{(l)}\right), \qquad \tilde A = A + I, \;\tilde D_{ii} = \sum_j\tilde A_{ij}$$
(first-order Chebyshev approximation with the renormalization trick)


## 3. Message passing neural networks

$$m_v^{(l+1)} = \bigoplus_{u\in\mathcal N(v)}M_l(h_v^{(l)}, h_u^{(l)}, e_{uv}), \qquad h_v^{(l+1)} = U_l(h_v^{(l)}, m_v^{(l+1)})$$
with a permutation-invariant aggregator $\bigoplus$ (sum, mean, max).

**GraphSAGE:** $h_v' = \sigma\left(W\cdot[h_v\,\|\,\text{AGG}(\{h_u: u\in\mathcal N(v)\})]\right)$, then $\ell_2$-normalize; neighbour sampling for scale.

**GAT:** $e_{vu} = \text{LeakyReLU}(a^\top[Wh_v\|Wh_u])$, $\;\alpha_{vu} = \text{softmax}_u(e_{vu})$, $\;h_v' = \sigma\left(\sum_u\alpha_{vu}Wh_u\right)$, with multi-head concatenation or averaging

**GIN:** $h_v' = \text{MLP}\left((1 + \epsilon)h_v + \sum_{u\in\mathcal N(v)}h_u\right)$. As expressive as the 1-Weisfeiler-Lehman test (the upper bound for standard MPNNs).

**Readout for graph-level tasks:** $h_G = \text{READOUT}(\{h_v\})$ (sum/mean/max pooling, Set2Set, attention pooling)


## 4. Known limitations

**Over-smoothing:** repeated averaging makes node features converge; the Dirichlet energy $E(X) = \text{tr}(X^\top LX)$ decays with depth.

**Over-squashing:** exponentially many neighbours squeezed into fixed-size vectors; measured by the Jacobian $\left|\frac{\partial h_v^{(L)}}{\partial x_u}\right|$ and related to graph curvature and effective resistance.

Fixes: residual connections, jumping knowledge, graph rewiring, virtual nodes, graph transformers.


## 5. Graph transformers and positional encodings

Full attention over nodes plus structural encodings: Laplacian eigenvectors (sign-ambiguous; use SignNet or random sign flips), random-walk encodings $\text{RW}_{ii}^k$ = diagonal of $(D^{-1}A)^k$, shortest-path-distance attention biases (Graphormer).


## 6. Link prediction and knowledge graphs

**Graph autoencoder:** $\hat A = \sigma(ZZ^\top)$, $Z = \text{GCN}(X, A)$; VGAE adds a KL term.

**Knowledge-graph embeddings** for triples $(h, r, t)$:
- TransE: $\|h + r - t\|$
- DistMult: $\langle h, r, t\rangle = \sum_ih_ir_it_i$
- ComplEx: $\text{Re}(\langle h, r, \bar t\rangle)$ in $\mathbb C^d$
- RotatE: $\|h\circ r - t\|$, $|r_i| = 1$

Margin ranking loss $\max(0, \gamma + d(h,r,t) - d(h', r, t'))$ with corrupted triples.

**Node2vec / DeepWalk:** skip-gram on random walks; node2vec biases walks with return parameter $p$ and in-out parameter $q$.


## 7. Spatiotemporal graph networks

**Diffusion convolution (DCRNN):** $X_{:,p}\star_\mathcal G f_\theta = \sum_{k=0}^{K-1}\left(\theta_{k,1}(D_O^{-1}W)^k + \theta_{k,2}(D_I^{-1}W^\top)^k\right)X_{:,p}$, used inside a GRU for traffic forecasting

**STGCN:** sandwiches of temporal gated convolutions and spectral graph convolutions

**Graph WaveNet:** learns an adaptive adjacency $\tilde A = \text{softmax}(\text{ReLU}(E_1E_2^\top))$ plus dilated causal convolutions

**Mesh-based weather models (GraphCast-style):** encode the grid to a multi-scale icosahedral mesh, run message passing on the mesh, decode back to the grid; trained on multi-step rollout MSE weighted by latitude ($\cos\phi$)


## 8. Geometric deep learning: symmetry and equivariance

A layer $f$ is **equivariant** to a group $G$ if $f(\rho_{in}(g)x) = \rho_{out}(g)f(x)$, and **invariant** if $f(\rho(g)x) = f(x)$.

| Domain | Symmetry group | Architecture |
|---|---|---|
| Grids / images | translations | CNNs |
| Sets | permutations $S_n$ | Deep Sets $\rho\left(\sum\phi(x_i)\right)$, transformers without PE |
| Graphs | permutations | GNNs |
| Spheres | rotations $SO(3)$ | spherical CNNs |
| 3D point clouds / molecules | $E(3)$, $SE(3)$ | EGNN, SE(3)-transformers, tensor field networks |

**E(n)-equivariant GNN (EGNN):**
$$m_{ij} = \phi_e(h_i, h_j, \|x_i - x_j\|^2, a_{ij}), \quad x_i' = x_i + C\sum_{j\ne i}(x_i - x_j)\phi_x(m_{ij}), \quad h_i' = \phi_h\left(h_i, \sum_jm_{ij}\right)$$

**Group convolution:** $(f * \psi)(g) = \int_Gf(h)\psi(g^{-1}h)\,dh$

**Spherical harmonics** $Y_\ell^m$ and **Clebsch-Gordan** tensor products build $SO(3)$-equivariant features of degree $\ell$.


## 9. Neural operators (functions to functions)

**DeepONet:** $G(u)(y)\approx\sum_{k=1}^pb_k(u(x_1),\dots,u(x_m))\,t_k(y)$ (branch × trunk networks)

**Fourier neural operator:** $v_{t+1}(x) = \sigma\left(Wv_t(x) + \mathcal F^{-1}\left(R_\phi\cdot\mathcal F(v_t)\right)(x)\right)$, keeping only the lowest $k$ Fourier modes; discretization-invariant.

**Physics-informed neural networks (PINNs):** $\mathcal L = \mathcal L_{data} + \lambda\frac1N\sum_i\left|\mathcal N[u_\theta](x_i)\right|^2 + \mathcal L_{BC/IC}$, where $\mathcal N$ is the PDE residual computed with autodiff.
